# 🚀 MultiSocial++: Efficient Multilingual Social-Media AI-Text Detection
### Integrated PDC + NLP Project | Aimal Khan & Awais Arshad

---
**Before running:** Make sure you have:
- ✅ Enabled GPU: Runtime → Change runtime type → T4 GPU
- ✅ Uploaded dataset to Google Drive in a folder called `MultiSocial_Project`
---

## Project Pipeline
```
Dataset → Parallel Preprocessing → Baseline (TF-IDF+LR) → XLM-R Fine-tuning
        → PDC Optimizations → ONNX Export → Evaluation & Ablation Study
```

---
# SECTION 0: Setup & Installation
Run this cell first. It installs everything needed for the project.

In [ ]:
# ============================================================
# CELL 0.1 — Install all required libraries
# ============================================================
!pip install -q transformers==4.40.0 datasets accelerate
!pip install -q onnx onnxruntime optimum
!pip install -q scikit-learn pandas numpy matplotlib seaborn tqdm
!pip install -q torch torchvision

print("✅ All libraries installed successfully!")

In [ ]:
# ============================================================
# CELL 0.2 — Mount Google Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os

# Set paths — CHANGE folder name here if yours is different
DRIVE_FOLDER  = '/content/drive/MyDrive/MultiSocial_Project'
DATA_FILE     = os.path.join(DRIVE_FOLDER, 'multisocial_anonymized.csv.gz')
OUTPUT_FOLDER = os.path.join(DRIVE_FOLDER, 'outputs')
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

print(f"📁 Drive mounted.")
print(f"📂 Dataset path : {DATA_FILE}")
print(f"📂 Outputs saved: {OUTPUT_FOLDER}")
print(f"✅ File exists  : {os.path.exists(DATA_FILE)}")

In [ ]:
# ============================================================
# CELL 0.3 — Global Imports & GPU Check
# ============================================================
import pandas as pd
import numpy as np
import time
import re
import multiprocessing
import warnings
import json
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from functools import partial
from concurrent.futures import ProcessPoolExecutor

import torch
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score, accuracy_score, classification_report

from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding
)
from datasets import Dataset as HFDataset

warnings.filterwarnings('ignore')

# GPU check
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️  Device        : {DEVICE}")
if torch.cuda.is_available():
    print(f"🎮  GPU Name      : {torch.cuda.get_device_name(0)}")
    print(f"💾  GPU Memory    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"🧵  CPU Cores     : {multiprocessing.cpu_count()}")

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
print("\n✅ Setup complete!")

---
# SECTION 1: Load & Explore the Dataset (EDA)

In [ ]:
# ============================================================
# CELL 1.1 — Load dataset
# ============================================================
print("📥 Loading dataset...")
t0 = time.time()

df = pd.read_csv(DATA_FILE, compression='gzip')

print(f"✅ Loaded in {time.time()-t0:.1f}s")
print(f"📊 Shape: {df.shape}")
print(f"\n📋 Columns: {list(df.columns)}")
print(f"\n🔍 First 3 rows:")
df.head(3)

In [ ]:
# ============================================================
# CELL 1.2 — Understand the label column
# ============================================================
# Find which column is the label (human vs AI)
print("📊 Unique values in each column:")
for col in df.columns:
    unique_vals = df[col].nunique()
    print(f"  {col}: {unique_vals} unique values → {list(df[col].unique()[:8])}")

In [ ]:
# ============================================================
# CELL 1.3 — Set column names based on what you see above
# ============================================================

# ---------- EDIT THESE if columns have different names ----------
TEXT_COL     = 'text'        # column with the actual post text
LABEL_COL    = 'label'       # column with 'human' / model name
LANG_COL     = 'language'    # column with language code
PLATFORM_COL = 'source'      # column with platform name
NOISE_COL    = 'potential_noise'  # 0/1 noise flag
# ---------------------------------------------------------------

# Verify columns exist
for col in [TEXT_COL, LABEL_COL, LANG_COL, PLATFORM_COL]:
    if col in df.columns:
        print(f"✅ Found column: '{col}'")
    else:
        print(f"❌ MISSING column: '{col}' — update column name above!")

In [ ]:
# ============================================================
# CELL 1.4 — Create binary label: human=0, AI=1
# ============================================================
df['binary_label'] = df[LABEL_COL].apply(lambda x: 0 if str(x).lower() == 'human' else 1)

print("📊 Binary label distribution:")
counts = df['binary_label'].value_counts()
print(f"  Human (0): {counts.get(0, 0):,}")
print(f"  AI    (1): {counts.get(1, 0):,}")
print(f"  Total    : {len(df):,}")

In [ ]:
# ============================================================
# CELL 1.5 — Exploratory Data Analysis plots
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('MultiSocial Dataset — Exploratory Data Analysis', fontsize=14, fontweight='bold')

# Plot 1: Label distribution
df['binary_label'].value_counts().plot(
    kind='bar', ax=axes[0,0], color=['#2196F3','#F44336'], edgecolor='black'
)
axes[0,0].set_title('Human vs AI Distribution')
axes[0,0].set_xticklabels(['Human','AI'], rotation=0)
axes[0,0].set_ylabel('Count')

# Plot 2: Platform distribution
df[PLATFORM_COL].value_counts().plot(
    kind='bar', ax=axes[0,1], color='#4CAF50', edgecolor='black'
)
axes[0,1].set_title('Posts per Platform')
axes[0,1].tick_params(axis='x', rotation=45)

# Plot 3: Top 10 languages
df[LANG_COL].value_counts().head(10).plot(
    kind='barh', ax=axes[1,0], color='#FF9800', edgecolor='black'
)
axes[1,0].set_title('Top 10 Languages')

# Plot 4: Text length distribution
df['text_len'] = df[TEXT_COL].astype(str).apply(lambda x: len(x.split()))
axes[1,1].hist(df['text_len'].clip(upper=200), bins=50, color='#9C27B0', edgecolor='black')
axes[1,1].set_title('Text Length Distribution (words)')
axes[1,1].set_xlabel('Word Count')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_FOLDER, 'eda_plots.png'), dpi=150)
plt.show()
print("✅ EDA plots saved!")

---
# SECTION 2: ⚡ PDC Contribution #1 — Parallel Preprocessing Pipeline

**What we're doing:** Compare SEQUENTIAL preprocessing vs PARALLEL preprocessing and measure the speedup.
This is a core PDC contribution of our project.

In [ ]:
# ============================================================
# CELL 2.1 — Define preprocessing functions
# ============================================================

def clean_text_single(text):
    """Clean one text: lowercase, remove URLs, normalize whitespace."""
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '[URL]', text)          # URLs
    text = re.sub(r'@\w+', '[USER]', text)                    # @mentions
    text = re.sub(r'#(\w+)', r'\1', text)                     # hashtags (keep word)
    text = re.sub(r'[^\w\s\[\]]', ' ', text)                  # special chars
    text = re.sub(r'\s+', ' ', text).strip()                  # extra spaces
    return text


# --- Sequential preprocessing (the slow baseline) ---
def preprocess_sequential(texts):
    """Process texts one by one — no parallelism."""
    return [clean_text_single(t) for t in texts]


# --- Parallel preprocessing using multiprocessing (our PDC contribution) ---
def preprocess_parallel(texts, n_workers=None):
    """Process texts using all CPU cores simultaneously."""
    if n_workers is None:
        n_workers = multiprocessing.cpu_count()
    
    with ProcessPoolExecutor(max_workers=n_workers) as executor:
        results = list(executor.map(clean_text_single, texts, chunksize=1000))
    return results


print(f"✅ Preprocessing functions defined.")
print(f"🧵  Will use {multiprocessing.cpu_count()} CPU cores for parallel processing.")

# Quick test
sample = "Check out https://example.com @user123 #AI is amazing!!!"
print(f"\n🔍 Sample before: {sample}")
print(f"✨ Sample after : {clean_text_single(sample)}")

In [ ]:
# ============================================================
# CELL 2.2 — BENCHMARK: Sequential vs Parallel (PDC Measurement)
# ============================================================
# Use a large sample to get meaningful timing results
BENCHMARK_SIZE = min(50000, len(df))
sample_texts = df[TEXT_COL].astype(str).tolist()[:BENCHMARK_SIZE]

print(f"🏁 Benchmarking preprocessing on {BENCHMARK_SIZE:,} texts...\n")

# --- Sequential ---
print("⏳ Running SEQUENTIAL preprocessing...")
t_start = time.time()
seq_results = preprocess_sequential(sample_texts)
t_seq = time.time() - t_start
print(f"✅ Sequential done: {t_seq:.2f} seconds")

# --- Parallel ---
print("\n⚡ Running PARALLEL preprocessing...")
t_start = time.time()
par_results = preprocess_parallel(sample_texts)
t_par = time.time() - t_start
print(f"✅ Parallel done: {t_par:.2f} seconds")

# --- Results ---
speedup = t_seq / t_par
print(f"\n{'='*50}")
print(f"📊 PREPROCESSING BENCHMARK RESULTS")
print(f"{'='*50}")
print(f"  Sequential time : {t_seq:.2f}s")
print(f"  Parallel time   : {t_par:.2f}s")
print(f"  🚀 Speedup      : {speedup:.2f}x")
print(f"  CPU cores used  : {multiprocessing.cpu_count()}")
print(f"{'='*50}")

# Save results for paper
benchmark_results = {
    'preprocessing': {
        'sequential_sec': round(t_seq, 3),
        'parallel_sec': round(t_par, 3),
        'speedup': round(speedup, 3),
        'texts_processed': BENCHMARK_SIZE,
        'cpu_cores': multiprocessing.cpu_count()
    }
}

with open(os.path.join(OUTPUT_FOLDER, 'benchmark_results.json'), 'w') as f:
    json.dump(benchmark_results, f, indent=2)
print("\n💾 Benchmark results saved!")

In [ ]:
# ============================================================
# CELL 2.3 — Apply parallel preprocessing to full dataset + cache
# ============================================================
CACHE_FILE = os.path.join(OUTPUT_FOLDER, 'preprocessed_cache.pkl')

if os.path.exists(CACHE_FILE):
    print("📂 Cache found! Loading preprocessed data...")
    df_clean = pd.read_pickle(CACHE_FILE)
    print("✅ Loaded from cache — skipping preprocessing!")
else:
    print(f"⚡ Preprocessing full dataset ({len(df):,} texts) in parallel...")
    t0 = time.time()
    all_texts = df[TEXT_COL].astype(str).tolist()
    df['clean_text'] = preprocess_parallel(all_texts)
    df_clean = df.copy()
    df_clean.to_pickle(CACHE_FILE)
    print(f"✅ Done in {time.time()-t0:.1f}s — cached to disk for future runs!")

print(f"\n📊 Clean dataset shape: {df_clean.shape}")
print(f"🔍 Sample cleaned text:")
print(f"   BEFORE: {df.iloc[0][TEXT_COL][:100]}")
print(f"   AFTER : {df_clean.iloc[0]['clean_text'][:100]}")

---
# SECTION 3: Train/Validation/Test Split

In [ ]:
# ============================================================
# CELL 3.1 — Create reproducible splits (70/15/15)
# ============================================================
from sklearn.model_selection import train_test_split

texts = df_clean['clean_text'].tolist()
labels = df_clean['binary_label'].tolist()
languages = df_clean[LANG_COL].tolist()
platforms = df_clean[PLATFORM_COL].tolist()

# Split: 70% train, 15% val, 15% test
X_train, X_temp, y_train, y_temp, lang_train, lang_temp, plat_train, plat_temp = train_test_split(
    texts, labels, languages, platforms,
    test_size=0.30, random_state=SEED, stratify=labels
)
X_val, X_test, y_val, y_test, lang_val, lang_test, plat_val, plat_test = train_test_split(
    X_temp, y_temp, lang_temp, plat_temp,
    test_size=0.50, random_state=SEED, stratify=y_temp
)

print(f"📊 Data Split Summary:")
print(f"   Train : {len(X_train):>7,} samples ({100*len(X_train)/len(texts):.0f}%)")
print(f"   Val   : {len(X_val):>7,} samples ({100*len(X_val)/len(texts):.0f}%)")
print(f"   Test  : {len(X_test):>7,} samples ({100*len(X_test)/len(texts):.0f}%)")
print(f"\n✅ Splits created with seed={SEED} for full reproducibility.")

---
# SECTION 4: Baseline Model — TF-IDF + Logistic Regression

This is our **lightweight baseline**. It's fast and interpretable.

In [ ]:
# ============================================================
# CELL 4.1 — TF-IDF + Logistic Regression
# ============================================================
print("🔤 Training TF-IDF + Logistic Regression baseline...")

t0 = time.time()

# TF-IDF vectorizer
vectorizer = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=2
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf   = vectorizer.transform(X_val)
X_test_tfidf  = vectorizer.transform(X_test)

# Logistic Regression
lr_clf = LogisticRegression(max_iter=1000, C=1.0, random_state=SEED, n_jobs=-1)
lr_clf.fit(X_train_tfidf, y_train)

t_lr = time.time() - t0

# Evaluate
y_pred_lr = lr_clf.predict(X_test_tfidf)
f1_lr     = f1_score(y_test, y_pred_lr, average='macro')
acc_lr    = accuracy_score(y_test, y_pred_lr)

print(f"\n{'='*50}")
print(f"📊 TF-IDF + LR Results")
print(f"{'='*50}")
print(f"  Macro-F1    : {f1_lr:.4f}")
print(f"  Accuracy    : {acc_lr:.4f}")
print(f"  Train time  : {t_lr:.2f}s")
print(f"{'='*50}")

benchmark_results['tfidf_lr'] = {
    'macro_f1': round(f1_lr, 4),
    'accuracy': round(acc_lr, 4),
    'train_time_sec': round(t_lr, 3)
}

In [ ]:
# ============================================================
# CELL 4.2 — Per-language and per-platform analysis (TF-IDF)
# ============================================================
print("📊 Per-Language Macro-F1 (TF-IDF+LR):")

lang_f1 = {}
unique_langs = list(set(lang_test))

for lang in unique_langs:
    idx = [i for i, l in enumerate(lang_test) if l == lang]
    if len(idx) < 10:
        continue
    y_true_l = [y_test[i] for i in idx]
    y_pred_l = [y_pred_lr[i] for i in idx]
    f1_l = f1_score(y_true_l, y_pred_l, average='macro', zero_division=0)
    lang_f1[lang] = round(f1_l, 4)

lang_f1_sorted = dict(sorted(lang_f1.items(), key=lambda x: x[1], reverse=True))
for lang, f1 in lang_f1_sorted.items():
    bar = '█' * int(f1 * 20)
    print(f"  {lang:5s} | {bar:<20} | {f1:.4f}")

benchmark_results['tfidf_lr']['per_language_f1'] = lang_f1_sorted

---
# SECTION 5: XLM-R Fine-tuning (Main Transformer Model)

In [ ]:
# ============================================================
# CELL 5.1 — Prepare HuggingFace Dataset for XLM-R
# ============================================================
MODEL_NAME = 'xlm-roberta-base'

print(f"📥 Loading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# For Colab free tier, use a manageable subset for fine-tuning
# Increase these if using Colab Pro or your groupmate's Ubuntu machine
MAX_TRAIN = 40000   # training samples
MAX_VAL   = 5000    # validation samples  
MAX_SEQ   = 128     # max token length (social posts are short)

def tokenize_fn(examples):
    return tokenizer(
        examples['text'],
        truncation=True,
        max_length=MAX_SEQ,
        padding=False  # DataCollator handles padding dynamically
    )

# Build HF datasets
train_dict = {'text': X_train[:MAX_TRAIN], 'label': y_train[:MAX_TRAIN]}
val_dict   = {'text': X_val[:MAX_VAL],     'label': y_val[:MAX_VAL]}
test_dict  = {'text': X_test,              'label': y_test}

hf_train = HFDataset.from_dict(train_dict)
hf_val   = HFDataset.from_dict(val_dict)
hf_test  = HFDataset.from_dict(test_dict)

# Tokenize in parallel using map()
print("⚡ Tokenizing datasets in parallel...")
t0 = time.time()
hf_train = hf_train.map(tokenize_fn, batched=True, num_proc=2, desc='Tokenizing train')
hf_val   = hf_val.map(tokenize_fn,   batched=True, num_proc=2, desc='Tokenizing val')
hf_test  = hf_test.map(tokenize_fn,  batched=True, num_proc=2, desc='Tokenizing test')
t_tokenize = time.time() - t0

print(f"\n✅ Tokenization complete in {t_tokenize:.1f}s")
print(f"   Train: {len(hf_train):,} | Val: {len(hf_val):,} | Test: {len(hf_test):,}")
benchmark_results['tokenization_sec'] = round(t_tokenize, 3)

In [ ]:
# ============================================================
# CELL 5.2 — Load XLM-R model
# ============================================================
print(f"📥 Loading model: {MODEL_NAME}")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: 'human', 1: 'ai'},
    label2id={'human': 0, 'ai': 1}
)
model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n📊 Model Parameters:")
print(f"   Total     : {total_params/1e6:.1f}M")
print(f"   Trainable : {trainable_params/1e6:.1f}M")
print(f"✅ Model loaded on {DEVICE}!")

In [ ]:
# ============================================================
# CELL 5.3 — Define compute_metrics for evaluation
# ============================================================
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    macro_f1 = f1_score(labels, preds, average='macro')
    accuracy = accuracy_score(labels, preds)
    return {'macro_f1': macro_f1, 'accuracy': accuracy}

In [ ]:
# ============================================================
# CELL 5.4 — BASELINE Training (no PDC optimizations)
#            We record training time to compare with optimized
# ============================================================
BASELINE_OUTPUT = os.path.join(OUTPUT_FOLDER, 'xlmr_baseline')
os.makedirs(BASELINE_OUTPUT, exist_ok=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

baseline_args = TrainingArguments(
    output_dir=BASELINE_OUTPUT,
    num_train_epochs=3,
    per_device_train_batch_size=16,     # baseline: no memory tricks
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    fp16=False,                          # ← NO mixed precision in baseline
    dataloader_num_workers=0,            # ← NO parallel dataloader in baseline
    seed=SEED,
    report_to='none',
    logging_steps=100,
    save_total_limit=1,
)

baseline_trainer = Trainer(
    model=model,
    args=baseline_args,
    train_dataset=hf_train,
    eval_dataset=hf_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("🏁 Training BASELINE XLM-R (no PDC optimizations)...")
print("   This will take ~15-25 minutes on Colab T4. Please wait.\n")

t0 = time.time()
baseline_trainer.train()
t_baseline_train = time.time() - t0

# Evaluate on test set
baseline_preds = baseline_trainer.predict(hf_test)
y_pred_baseline = np.argmax(baseline_preds.predictions, axis=-1)
f1_baseline  = f1_score(y_test, y_pred_baseline, average='macro')
acc_baseline = accuracy_score(y_test, y_pred_baseline)

print(f"\n{'='*50}")
print(f"📊 BASELINE XLM-R Results")
print(f"{'='*50}")
print(f"  Macro-F1   : {f1_baseline:.4f}")
print(f"  Accuracy   : {acc_baseline:.4f}")
print(f"  Train time : {t_baseline_train:.1f}s ({t_baseline_train/60:.1f} min)")
print(f"{'='*50}")

benchmark_results['xlmr_baseline'] = {
    'macro_f1': round(f1_baseline, 4),
    'accuracy': round(acc_baseline, 4),
    'train_time_sec': round(t_baseline_train, 2)
}

# Save model
baseline_trainer.save_model(BASELINE_OUTPUT)
print("💾 Baseline model saved!")

---
# SECTION 6: ⚡ PDC Contribution #2 — Optimized Training

We now re-train with **mixed precision (fp16)**, **parallel dataloading**, and **gradient accumulation**.
Then measure how much faster it is vs the baseline.

In [ ]:
# ============================================================
# CELL 6.1 — PDC-Optimized Training
# ============================================================
# Reload fresh model weights for fair comparison
model_opt = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2,
    id2label={0:'human',1:'ai'}, label2id={'human':0,'ai':1}
)
model_opt.to(DEVICE)

OPT_OUTPUT = os.path.join(OUTPUT_FOLDER, 'xlmr_optimized')
os.makedirs(OPT_OUTPUT, exist_ok=True)

optimized_args = TrainingArguments(
    output_dir=OPT_OUTPUT,
    num_train_epochs=3,
    per_device_train_batch_size=32,        # ← LARGER batch (memory-aware)
    per_device_eval_batch_size=64,
    gradient_accumulation_steps=2,         # ← PDC: gradient accumulation
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    fp16=True,                             # ← PDC: mixed precision training
    dataloader_num_workers=2,              # ← PDC: parallel data loading
    dataloader_pin_memory=True,            # ← PDC: faster CPU→GPU transfer
    seed=SEED,
    report_to='none',
    logging_steps=100,
    save_total_limit=1,
)

optimized_trainer = Trainer(
    model=model_opt,
    args=optimized_args,
    train_dataset=hf_train,
    eval_dataset=hf_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("🚀 Training OPTIMIZED XLM-R (with all PDC improvements)...")
print("   PDC Optimizations active:")
print("   ✅ fp16 Mixed Precision")
print("   ✅ Parallel DataLoader (num_workers=2)")
print("   ✅ Pin Memory (faster GPU transfer)")
print("   ✅ Gradient Accumulation (effective batch=64)")
print("   ✅ Memory-aware larger batch size")
print("\n   Training...\n")

t0 = time.time()
optimized_trainer.train()
t_opt_train = time.time() - t0

# Evaluate on test set
opt_preds = optimized_trainer.predict(hf_test)
y_pred_opt = np.argmax(opt_preds.predictions, axis=-1)
f1_opt  = f1_score(y_test, y_pred_opt, average='macro')
acc_opt = accuracy_score(y_test, y_pred_opt)

training_speedup = t_baseline_train / t_opt_train
f1_drop = f1_baseline - f1_opt

print(f"\n{'='*55}")
print(f"📊 PDC-OPTIMIZED XLM-R Results")
print(f"{'='*55}")
print(f"  Macro-F1          : {f1_opt:.4f}")
print(f"  Accuracy          : {acc_opt:.4f}")
print(f"  Train time        : {t_opt_train:.1f}s ({t_opt_train/60:.1f} min)")
print(f"  ─────────────────────────────────────────")
print(f"  🚀 Training Speedup: {training_speedup:.2f}x faster than baseline")
print(f"  📉 Macro-F1 drop   : {f1_drop:.4f} (target: ≤ 1.0)")
print(f"{'='*55}")

benchmark_results['xlmr_optimized'] = {
    'macro_f1': round(f1_opt, 4),
    'accuracy': round(acc_opt, 4),
    'train_time_sec': round(t_opt_train, 2),
    'speedup_vs_baseline': round(training_speedup, 3),
    'f1_drop': round(f1_drop, 4)
}

optimized_trainer.save_model(OPT_OUTPUT)
print("💾 Optimized model saved!")

---
# SECTION 7: ⚡ PDC Contribution #3 — ONNX Export + INT8 Quantization (Fast Inference)

In [ ]:
# ============================================================
# CELL 7.1 — Export model to ONNX format
# ============================================================
from optimum.onnxruntime import ORTModelForSequenceClassification
from optimum.onnxruntime.configuration import AutoQuantizationConfig
from optimum.onnxruntime import ORTQuantizer
import onnxruntime as ort

ONNX_OUTPUT = os.path.join(OUTPUT_FOLDER, 'xlmr_onnx')

print("📦 Exporting model to ONNX...")
t0 = time.time()

ort_model = ORTModelForSequenceClassification.from_pretrained(
    OPT_OUTPUT,
    export=True
)
ort_model.save_pretrained(ONNX_OUTPUT)
tokenizer.save_pretrained(ONNX_OUTPUT)

print(f"✅ ONNX export done in {time.time()-t0:.1f}s")
print(f"   Saved to: {ONNX_OUTPUT}")

In [ ]:
# ============================================================
# CELL 7.2 — INT8 Quantization (further compression)
# ============================================================
QUANT_OUTPUT = os.path.join(OUTPUT_FOLDER, 'xlmr_quantized')

print("🔢 Applying INT8 quantization...")
t0 = time.time()

quantizer = ORTQuantizer.from_pretrained(ONNX_OUTPUT)
qconfig = AutoQuantizationConfig.avx512_vnni(is_static=False, per_channel=False)

quantizer.quantize(
    save_dir=QUANT_OUTPUT,
    quantization_config=qconfig
)

print(f"✅ Quantization done in {time.time()-t0:.1f}s")

# Compare model file sizes
orig_size  = os.path.getsize(os.path.join(ONNX_OUTPUT, 'model.onnx')) / 1e6
quant_size = os.path.getsize(os.path.join(QUANT_OUTPUT, 'model_quantized.onnx')) / 1e6
print(f"\n📦 Model Size Comparison:")
print(f"   Original ONNX  : {orig_size:.1f} MB")
print(f"   Quantized INT8 : {quant_size:.1f} MB")
print(f"   Size reduction : {100*(1-quant_size/orig_size):.1f}%")

benchmark_results['model_sizes'] = {
    'onnx_mb': round(orig_size, 2),
    'quantized_mb': round(quant_size, 2),
    'size_reduction_pct': round(100*(1-quant_size/orig_size), 1)
}

In [ ]:
# ============================================================
# CELL 7.3 — BENCHMARK: Inference Throughput
# ============================================================
# Compare: PyTorch baseline vs ONNX vs ONNX Quantized

INFERENCE_TEXTS = X_test[:1000]  # 1000 texts for benchmarking
BATCH_SIZE = 32

def batch_inference_pytorch(texts, model, tokenizer, batch_size=32):
    """PyTorch batch inference — standard approach."""
    model.eval()
    all_preds = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            enc = tokenizer(batch, truncation=True, max_length=MAX_SEQ,
                           padding=True, return_tensors='pt').to(DEVICE)
            out = model(**enc)
            preds = torch.argmax(out.logits, dim=-1).cpu().tolist()
            all_preds.extend(preds)
    return all_preds


def batch_inference_onnx(texts, ort_model, tokenizer, batch_size=32):
    """ONNX batch inference — optimized runtime."""
    all_preds = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc = tokenizer(batch, truncation=True, max_length=MAX_SEQ,
                       padding=True, return_tensors='pt')
        out = ort_model(**enc)
        preds = out.logits.argmax(dim=-1).tolist()
        all_preds.extend(preds)
    return all_preds


# Load ONNX and quantized models
ort_infer = ORTModelForSequenceClassification.from_pretrained(ONNX_OUTPUT)
ort_quant = ORTModelForSequenceClassification.from_pretrained(
    QUANT_OUTPUT, file_name='model_quantized.onnx'
)

print(f"🏁 Inference Throughput Benchmark ({len(INFERENCE_TEXTS)} texts, batch={BATCH_SIZE})\n")

# 1. PyTorch baseline
print("⏳ PyTorch (baseline) inference...")
t0 = time.time()
preds_pt = batch_inference_pytorch(INFERENCE_TEXTS, model_opt, tokenizer, BATCH_SIZE)
t_pt = time.time() - t0
thr_pt = len(INFERENCE_TEXTS) / t_pt
print(f"   Time: {t_pt:.2f}s | Throughput: {thr_pt:.0f} texts/sec")

# 2. ONNX
print("⚡ ONNX inference...")
t0 = time.time()
preds_onnx = batch_inference_onnx(INFERENCE_TEXTS, ort_infer, tokenizer, BATCH_SIZE)
t_onnx = time.time() - t0
thr_onnx = len(INFERENCE_TEXTS) / t_onnx
print(f"   Time: {t_onnx:.2f}s | Throughput: {thr_onnx:.0f} texts/sec")

# 3. ONNX Quantized
print("⚡ ONNX Quantized (INT8) inference...")
t0 = time.time()
preds_quant = batch_inference_onnx(INFERENCE_TEXTS, ort_quant, tokenizer, BATCH_SIZE)
t_quant = time.time() - t0
thr_quant = len(INFERENCE_TEXTS) / t_quant
print(f"   Time: {t_quant:.2f}s | Throughput: {thr_quant:.0f} texts/sec")

# Speedups
speedup_onnx  = thr_onnx / thr_pt
speedup_quant = thr_quant / thr_pt

# F1 for each inference variant
true_labels = y_test[:len(INFERENCE_TEXTS)]
f1_pt    = f1_score(true_labels, preds_pt,   average='macro')
f1_onnx  = f1_score(true_labels, preds_onnx, average='macro')
f1_quant = f1_score(true_labels, preds_quant, average='macro')

print(f"\n{'='*60}")
print(f"📊 INFERENCE BENCHMARK RESULTS")
print(f"{'='*60}")
print(f"  {'Method':<25} {'Throughput':>12} {'Speedup':>9} {'Macro-F1':>10}")
print(f"  {'-'*57}")
print(f"  {'PyTorch (baseline)':<25} {thr_pt:>10.0f}/s {'1.00x':>9} {f1_pt:>10.4f}")
print(f"  {'ONNX':<25} {thr_onnx:>10.0f}/s {speedup_onnx:>8.2f}x {f1_onnx:>10.4f}")
print(f"  {'ONNX + INT8 Quant':<25} {thr_quant:>10.0f}/s {speedup_quant:>8.2f}x {f1_quant:>10.4f}")
print(f"{'='*60}")

benchmark_results['inference'] = {
    'pytorch': {'throughput_per_sec': round(thr_pt), 'macro_f1': round(f1_pt, 4)},
    'onnx':    {'throughput_per_sec': round(thr_onnx), 'speedup': round(speedup_onnx, 3), 'macro_f1': round(f1_onnx, 4)},
    'onnx_int8': {'throughput_per_sec': round(thr_quant), 'speedup': round(speedup_quant, 3), 'macro_f1': round(f1_quant, 4)}
}

---
# SECTION 8: Ablation Study & Final Evaluation

In [ ]:
# ============================================================
# CELL 8.1 — Full evaluation: per-language & per-platform
# ============================================================
# Use optimized model predictions on full test set
y_pred_final = y_pred_opt   # from optimized XLM-R

print("📊 Full Classification Report (Optimized XLM-R):")
print(classification_report(y_test, y_pred_final, target_names=['human','ai']))

# Per-language breakdown
print("\n📊 Per-Language Macro-F1 (Optimized XLM-R):")
lang_f1_xlmr = {}
for lang in set(lang_test):
    idx = [i for i,l in enumerate(lang_test) if l == lang]
    if len(idx) < 10: continue
    y_t = [y_test[i] for i in idx]
    y_p = [y_pred_final[i] for i in idx]
    f1_l = f1_score(y_t, y_p, average='macro', zero_division=0)
    lang_f1_xlmr[lang] = round(f1_l, 4)

for lang, f1 in sorted(lang_f1_xlmr.items(), key=lambda x: x[1], reverse=True):
    bar = '█' * int(f1 * 20)
    print(f"  {lang:5s} | {bar:<20} | {f1:.4f}")

# Per-platform breakdown
print("\n📊 Per-Platform Macro-F1 (Optimized XLM-R):")
plat_f1_xlmr = {}
for plat in set(plat_test):
    idx = [i for i,p in enumerate(plat_test) if p == plat]
    if len(idx) < 10: continue
    y_t = [y_test[i] for i in idx]
    y_p = [y_pred_final[i] for i in idx]
    f1_p = f1_score(y_t, y_p, average='macro', zero_division=0)
    plat_f1_xlmr[plat] = round(f1_p, 4)
    print(f"  {str(plat):15s} | F1: {f1_p:.4f}")

benchmark_results['xlmr_optimized']['per_language_f1'] = lang_f1_xlmr
benchmark_results['xlmr_optimized']['per_platform_f1'] = plat_f1_xlmr

In [ ]:
# ============================================================
# CELL 8.2 — Ablation Study Table
# ============================================================
print("\n" + "="*70)
print("📊 ABLATION STUDY — MultiSocial++")
print("="*70)
print(f"  {'Configuration':<35} {'Macro-F1':>10} {'Train Time':>12} {'Inf Speed':>12}")
print(f"  {'-'*67}")
print(f"  {'TF-IDF + LR (Lightweight)':<35} {f1_lr:>10.4f} {benchmark_results['tfidf_lr']['train_time_sec']:>10.1f}s {'N/A':>12}")
print(f"  {'XLM-R Baseline (no PDC)':<35} {f1_baseline:>10.4f} {t_baseline_train:>10.1f}s {thr_pt:>10.0f}/s")
print(f"  {'+ Mixed Precision (fp16)':<35} {'(incl. below)':>10} {'':>12} {'':>12}")
print(f"  {'+ Parallel DataLoader':<35} {'(incl. below)':>10} {'':>12} {'':>12}")
print(f"  {'+ Grad. Accumulation':<35} {'(incl. below)':>10} {'':>12} {'':>12}")
print(f"  {'XLM-R + All PDC Opts':<35} {f1_opt:>10.4f} {t_opt_train:>10.1f}s {thr_pt:>10.0f}/s")
print(f"  {'+ ONNX Export':<35} {f1_onnx:>10.4f} {'':>12} {thr_onnx:>10.0f}/s")
print(f"  {'+ ONNX + INT8 Quant (Full++)':<35} {f1_quant:>10.4f} {'':>12} {thr_quant:>10.0f}/s")
print("="*70)
print(f"\n🚀 KEY RESULTS:")
print(f"   Preprocessing speedup : {benchmark_results['preprocessing']['speedup']:.2f}x (target: ≥2x)")
print(f"   Training speedup      : {training_speedup:.2f}x (target: ≥25% faster)")
print(f"   Inference speedup     : {speedup_quant:.2f}x (target: ≥2x)")
print(f"   Max F1 drop (quant)   : {f1_opt-f1_quant:.4f} (target: ≤1.0)")

In [ ]:
# ============================================================
# CELL 8.3 — Final Summary Visualization
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('MultiSocial++ — Results Summary', fontsize=14, fontweight='bold')

# Plot 1: Macro-F1 comparison
methods = ['TF-IDF\n+LR', 'XLM-R\nBaseline', 'XLM-R\nOptimized', 'ONNX', 'ONNX\nINT8']
f1s     = [f1_lr, f1_baseline, f1_opt, f1_onnx, f1_quant]
colors  = ['#90A4AE', '#42A5F5', '#66BB6A', '#FFA726', '#EF5350']
bars = axes[0].bar(methods, f1s, color=colors, edgecolor='black')
axes[0].set_title('Macro-F1 Score')
axes[0].set_ylim(0, 1.0)
axes[0].set_ylabel('Macro-F1')
for bar, f1 in zip(bars, f1s):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                 f'{f1:.3f}', ha='center', fontsize=9, fontweight='bold')

# Plot 2: Training speedup
train_methods = ['Baseline\n(no PDC)', 'Optimized\n(all PDC)']
train_times   = [t_baseline_train/60, t_opt_train/60]
axes[1].bar(train_methods, train_times, color=['#EF5350','#66BB6A'], edgecolor='black')
axes[1].set_title(f'Training Time (Speedup: {training_speedup:.2f}x)')
axes[1].set_ylabel('Minutes')
for i, t in enumerate(train_times):
    axes[1].text(i, t+0.2, f'{t:.1f} min', ha='center', fontweight='bold')

# Plot 3: Inference throughput
inf_methods = ['PyTorch', 'ONNX', 'ONNX+INT8']
throughputs  = [thr_pt, thr_onnx, thr_quant]
axes[2].bar(inf_methods, throughputs, color=['#42A5F5','#FFA726','#66BB6A'], edgecolor='black')
axes[2].set_title(f'Inference Throughput (texts/sec)')
axes[2].set_ylabel('Texts / Second')
for i, t in enumerate(throughputs):
    axes[2].text(i, t+5, f'{t:.0f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_FOLDER, 'results_summary.png'), dpi=150)
plt.show()
print("✅ Results visualization saved!")

In [ ]:
# ============================================================
# CELL 8.4 — Save all benchmark results
# ============================================================
with open(os.path.join(OUTPUT_FOLDER, 'benchmark_results.json'), 'w') as f:
    json.dump(benchmark_results, f, indent=2)

print("💾 All results saved to benchmark_results.json")
print("\n📋 Final benchmark_results:")
print(json.dumps(benchmark_results, indent=2))

---
# SECTION 9: Run a Single Text Prediction (Demo for Viva)

Use this to demonstrate your model live during the presentation!

In [ ]:
# ============================================================
# CELL 9.1 — Interactive demo: classify any text
# ============================================================

def predict_text(text, model, tokenizer, device=DEVICE):
    """Predict whether a text is human-written or AI-generated."""
    model.eval()
    cleaned = clean_text_single(text)
    enc = tokenizer(cleaned, return_tensors='pt', truncation=True,
                    max_length=MAX_SEQ, padding=True).to(device)
    with torch.no_grad():
        out = model(**enc)
    prob = torch.softmax(out.logits, dim=-1).cpu().numpy()[0]
    pred = int(np.argmax(prob))
    label = 'AI-Generated 🤖' if pred == 1 else 'Human-Written 👤'
    print(f"\n📝 Input: {text[:100]}")
    print(f"🔍 Prediction : {label}")
    print(f"   Confidence : {prob[pred]*100:.1f}%")
    print(f"   Human prob : {prob[0]*100:.1f}% | AI prob: {prob[1]*100:.1f}%")
    return pred


# Try some examples
print("=" * 55)
print("🎮 DEMO: AI-Text Detector")
print("=" * 55)

# Example 1: Likely AI
predict_text(
    "As an AI language model, I can provide comprehensive information about climate change and its multifaceted impacts on global ecosystems.",
    model_opt, tokenizer
)

# Example 2: Likely Human
predict_text(
    "omg just saw the craziest thing on my way to work lmaooo why is this my life 😭 #mondaymorning",
    model_opt, tokenizer
)

# Example 3: Try your own!
YOUR_TEXT = "Type any social media post here to test!"
predict_text(YOUR_TEXT, model_opt, tokenizer)

---
# ✅ Project Complete!

## Summary of What We've Built:

| Component | Status | Contribution |
|-----------|--------|-------------|
| Dataset loading & EDA | ✅ | Understanding the data |
| **Parallel preprocessing** | ✅ | **PDC #1** — multiprocessing speedup |
| TF-IDF + LR baseline | ✅ | Lightweight baseline |
| XLM-R fine-tuning | ✅ | Main multilingual model |
| **Mixed precision + parallel dataloader** | ✅ | **PDC #2** — training speedup |
| **ONNX export + INT8 quantization** | ✅ | **PDC #3** — inference speedup |
| Ablation study | ✅ | Attribution of each gain |
| Per-language & per-platform analysis | ✅ | Robustness evaluation |
| Live demo | ✅ | For viva presentation |

## Files saved to your Google Drive:
- `outputs/eda_plots.png` — EDA visualization
- `outputs/results_summary.png` — Results charts
- `outputs/benchmark_results.json` — All numbers for the paper
- `outputs/preprocessed_cache.pkl` — Cached preprocessing
- `outputs/xlmr_baseline/` — Baseline model
- `outputs/xlmr_optimized/` — PDC-optimized model
- `outputs/xlmr_onnx/` — ONNX exported model
- `outputs/xlmr_quantized/` — INT8 quantized model